[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabanib/Clases_clasificacion/blob/main/04_random-forest-clasificacion.ipynb#copy=true)


[![Open in Anaconda](https://static.anaconda.cloud/content/a22d04e8445b700f28937ab3231b8cded505d0395c63b7a269696722196d5415)](https://anaconda.com/api/nbserve/launch_notebook?nb_url=https%3A%2F%2Fanaconda.com%2Fapi%2Fprojects%2F6905e11d-e422-418c-ad09-bff2871f164e%2Ffiles%2F04_random-forest-clasificacion.ipynb%3Fversion%3Dac7c43d2-9e40-4aa9-a0a9-0775b55380fe)





# Métodos de ensamble para clasificación

v1

<p>
Los métodos de ensamble en machine learning son técnicas que combinan múltiples modelos.
</li><br><br>
</ul>
<p>
Existen diferentes enfoques para construir ensambles en machine learning, siendo los más comunes:
<ul align="justify">
<li> <b>Bagging</b>: se ajustan múltiples modelos, cada uno con un subconjunto distinto de los datos de entrenamiento.  El ejemplo más popular de algoritmo de ensamble basados en Bagging es <b>Random Forest</b>.
<br><br>
<li> <b>Boosting</b>: a diferencia del bagging, se basa en construir una secuencia de modelos donde cada uno se enfoca en mejorar los errores del modelo anterior. Algunos ejemplos populares de algoritmos de Boosting son <b>AdaBoost, Gradiente Boosting Machine, XGBoost, LightGBM</b> y <b>CatBoost</b>. Cada algoritmo de Boosting tiene su propio enfoque para ajustar los pesos de las instancias y combinar las predicciones, pero todos comparten la idea general de construir una secuencia de modelos que se mejoren consecutivamente.

 # **Bagging**

<p>
El término <b>bagging</b> es el diminutivo de <b>bootstrap aggregation</b>, y hace referencia al empleo del muestreo aleatorio simple con reemplazo con el fin de reducir el error de algunos modelos de aprendizaje automático, como por ejemplo los basados en árboles.

# Datos

<p>
El dataset es un conjunto de datos de 20.640 registros de viviendas en California. Este es un conjunto de datos muy útil para entrenar modelos de aprendizaje automático para predecir el valor de la vivienda.


### Preparación del entorno

Se importan `pandas` para manipular tablas y `numpy` para operaciones numéricas. Estas bibliotecas se reutilizan en la preparación del objetivo, el muestreo bootstrap y el cálculo de métricas manuales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Carga e inspección

El archivo se lee desde su fuente. `head()` permite revisar ejemplos de registros e `info()` muestra tipos de datos y valores faltantes. Antes de modelar conviene comprobar que las columnas numéricas estén correctamente interpretadas y conocer cuántas filas se eliminan por datos incompletos.

In [ ]:
url = "https://raw.githubusercontent.com/pabanib/dataframes/master/housing.csv"

In [ ]:
datos = pd.read_csv(url)

In [ ]:
datos.head()

In [ ]:
datos.info()

Las filas incompletas se eliminan para que todos los estimadores reciban matrices sin valores faltantes. También se quitan coordenadas y la variable categórica geográfica para mantener el flujo numérico del ejemplo original. Esta simplificación facilita el foco en bagging, aunque en un proyecto real esas variables podrían contener información útil.

In [ ]:
datos.dropna(inplace=True)

In [ ]:
datos.drop(columns=["longitude","latitude","ocean_proximity"], inplace=True)
datos.head()

La variable original `median_house_value` es continua. Para formular un problema de clasificación, se crea `clase_valor`: **alto** cuando el valor de la vivienda es igual o superior a la mediana del conjunto y **bajo** en caso contrario. Usar la mediana produce, en general, clases de tamaño parecido y facilita evaluar el clasificador. Después se elimina la variable continua original para evitar fuga de información.

In [ ]:
umbral_valor = datos['median_house_value'].median()
datos['clase_valor'] = np.where(
    datos['median_house_value'] >= umbral_valor,
    'alto',
    'bajo'
)
datos.drop(columns='median_house_value', inplace=True)

print(f"Umbral de clasificación: {umbral_valor:,.0f}")
datos['clase_valor'].value_counts()

## Ejemplo con varias muestras

Vamos a definir una pequeña muestra del conjunto total que usaremos como datos de testeo para controlar las predicciones.

Se fija una semilla para hacer reproducible el muestreo y se reservan 100 observaciones como conjunto de prueba del ejemplo manual. Esas filas se usarán para comparar las reglas obtenidas a partir de diferentes muestras.

In [ ]:
np.random.seed(4533)
dato_test =  datos.sample(100)
dato_test

<p>
Supongamos tres muestras diferentes que se corresponden a los mismos datos poblacionales. Estas muestras son tomadas de manera aleatoria desde los datos de diez unidades cada una.

Se generan tres muestras pequeñas para evidenciar la variabilidad del aprendizaje. Como cada muestra contiene pocas observaciones, sus clases dominantes y reglas pueden cambiar notablemente. Este comportamiento motiva la combinación de múltiples modelos.

In [ ]:
datos_entrenamiento_manual = datos.drop(index=dato_test.index)
dato_ent1 = datos_entrenamiento_manual.sample(15)
dato_ent2 = datos_entrenamiento_manual.sample(15)
dato_ent3 = datos_entrenamiento_manual.sample(15)

In [ ]:
assert dato_test.index.intersection(dato_ent1.index).empty
assert dato_test.index.intersection(dato_ent2.index).empty
assert dato_test.index.intersection(dato_ent3.index).empty

Con cada muestra se construye un clasificador elemental de una sola regla. El ingreso mediano divide los datos en dos ramas y cada rama predice su clase mayoritaria. Aunque es una simplificación de un árbol real, permite ver por qué modelos entrenados con muestras distintas pueden producir reglas y predicciones diferentes

In [ ]:
def arbol_simple(df, variable='median_income'):
    umbral = df[variable].mean()
    clases_por_rama = (
        df.groupby(df[variable] > umbral)['clase_valor'].agg(lambda serie: serie.mode().iloc[0])
    )
    return {
        'umbral': umbral,
        'menor_igual': clases_por_rama.loc[False],
        'mayor': clases_por_rama.loc[True]
    }

arbol_simple(dato_ent1)

In [ ]:
arbol_simple(dato_ent2)

In [ ]:
arbol_simple(dato_ent3)

La función siguiente aplica las reglas aprendidas por el clasificador simple a cada observación de prueba. La salida es una secuencia de etiquetas `alto` o `bajo`, una por vivienda. Esta separación entre ajuste y predicción reproduce la lógica general de los estimadores de `scikit-learn`.

In [ ]:
def predecir(reglas, variable='median_income'):
    predicciones = []
    for valor in dato_test[variable]:
        if valor > reglas['umbral']:
            predicciones.append(reglas['mayor'])
        else:
            predicciones.append(reglas['menor_igual'])
    return np.array(predicciones)

predecir(arbol_simple(dato_ent1))

In [ ]:
predecir(arbol_simple(dato_ent2))

In [ ]:
predecir(arbol_simple(dato_ent3))

In [ ]:
df_resultados = dato_test[['median_income', 'clase_valor']].copy()
df_resultados['prediccion_1'] = predecir(arbol_simple(dato_ent1))
df_resultados['prediccion_2'] = predecir(arbol_simple(dato_ent2))
df_resultados['prediccion_3'] = predecir(arbol_simple(dato_ent3))
df_resultados

La exactitud es la proporción de observaciones correctamente clasificadas. Se calcula para cada uno de los tres modelos simples; las diferencias muestran la variabilidad que aparece al entrenar con muestras pequeñas.

In [ ]:
columnas_prediccion = ['prediccion_1', 'prediccion_2', 'prediccion_3']
exactitudes = {
    columna: (df_resultados['clase_valor'] == df_resultados[columna]).mean() for columna in columnas_prediccion
}
exactitudes

En clasificación, las predicciones no se promedian numéricamente: se combinan mediante **voto mayoritario**. Para cada vivienda se selecciona la clase más repetida entre los tres modelos y se calcula la exactitud del ensamble.

In [ ]:
voto_tres_modelos = df_resultados[columnas_prediccion].mode(axis=1)[0]
(df_resultados['clase_valor'] == voto_tres_modelos).mean()

In [ ]:
voto_tres_modelos.head(10)

El voto puede superar a algunos modelos individuales porque reduce la influencia de reglas inestables. Como referencia adicional, se ajusta el mismo clasificador simple con todos los datos y se compara su exactitud con la del ensamble.

In [ ]:
df_resultados['prediccion_total'] = predecir(arbol_simple(datos))
df_resultados

In [ ]:
arbol_simple(datos)

In [ ]:
(df_resultados['clase_valor'] == df_resultados['prediccion_total']).mean()

In [ ]:
data = datos.sample(1000)
color = data.clase_valor.apply(lambda x: 'red' if x == 'bajo' else 'green') 
data.plot(kind = 'scatter', x = 'median_income', y = 'housing_median_age', c = color
            ,alpha = 0.4
           )





En la práctica suele haber una sola muestra observada. El bootstrap permite crear muchas versiones de ella mediante muestreo con reemplazo y es la base del bagging. Cada remuestra puede generar un clasificador diferente; al combinar sus votos se reduce la variabilidad de la predicción.

## Bootstrap

Nos vamos a quedar solo con la primera muestra. Seguimos trabajando con pocos valores para fines explicativos.

In [ ]:
d1 = dato_ent1[['median_income', 'clase_valor']].copy()
d1

El **bootstrap** toma observaciones con reemplazo: algunas pueden aparecer varias veces y otras quedar fuera. Primero se observa cómo cambia entre remuestras la proporción de viviendas de valor alto. La distribución obtenida permite apreciar la incertidumbre de ese estadístico cuando la muestra es pequeña.

In [ ]:
(d1.sample(10, replace = True),d1.sample(10, replace = True),d1.sample(10, replace = True))

In [ ]:
(d1['clase_valor'] == 'alto').mean()

In [ ]:
proporciones_altas = []
for _ in range(1000):
    muestra_bootstrap = d1.sample(len(d1), replace=True)
    proporciones_altas.append((muestra_bootstrap['clase_valor'] == 'alto').mean())

pd.Series(proporciones_altas).plot(kind='hist', bins=11, color='0.65', edgecolor='black')

In [ ]:
np.mean(proporciones_altas)

# Random Forest para clasificación

Un **Random Forest** combina muchos árboles de decisión entrenados con muestras bootstrap y subconjuntos aleatorios de predictores. En clasificación, cada árbol emite una clase y el bosque decide por voto. La aleatoriedad reduce la correlación entre árboles y suele mejorar la estabilidad respecto de un árbol individual.

 ## **Procedimiento Random Forest**



<ul align="justify">
1. Se toma una muestra aleatoria simple con reemplazo del conjunto de datos original. Esta muestra se llama "bag" o "bolsa". La muestra tiene el mismo tamaño que el conjunto de datos original.
<br><br>
2. Se realiza una muestra aleatoria simple de las variables predictoras. Esto significa que se seleccionan aleatoriamente un subconjunto de variables predictoras del conjunto de variables disponibles.
<br><br>
3. Se construye un árbol de decisión utilizando la muestra de la bolsa y las variables predictoras seleccionadas en el paso anterior. El árbol se construye sin poda, es decir, sin limitar su crecimiento.
<br><br>
4. Se realizan las predicciones utilizando las observaciones que no se incluyeron en la bolsa, es decir, las observaciones que quedaron fuera de la muestra seleccionada. Estas predicciones se conocen como "out-of-bag".
<br><br>
5. Se repiten los pasos 1 al 4 tantas veces como árboles se deseen construir. Cada iteración genera una nueva muestra de la bolsa, una nueva selección de variables predictoras y un nuevo árbol de decisión.
<br><br>
6. Las predicciones de cada árbol se combinan para obtener una predicción final. En el caso de la regresión, esto implica promediar las predicciones de los modelos base. En la clasificación, se puede utilizar una votación mayoritaria para determinar la clase final.

 ## **Ventajas del Random Forest**

<ul p align="justify">
<li><b>Precisión</b>: son muy precisos, especialmente cuando hay muchos datos disponibles. Esto se debe a que combinan el juicio de muchos árboles de decisión, cada uno de los cuales es entrenado en un subconjunto diferente de los datos.
</li><br>
<li><b>Robustez</b>: son resistentes a los valores atípicos y al ruido en los datos. Esto se debe a que cada árbol de decisión solo ve una parte de los datos, por lo que no se ve afectado por valores atípicos o ruido.
</li><br>
<li><b>Capacidad para manejar grandes conjuntos de datos</b>: pueden manejar grandes conjuntos de datos sin problemas. Esto se debe a que se pueden entrenar en paralelo, lo que significa que se pueden entrenar muchos árboles de decisión al mismo tiempo.
</li><br>
<li><b>Interpretabilidad</b>: son relativamente fáciles de interpretar en comparación con otros algoritmos de aprendizaje automático como las redes neuronales. Esto se debe a que se pueden visualizar como un conjunto de árboles de decisión, cada uno de los cuales representa una forma diferente de clasificar los datos.
</li>
</ul>
<br>
<p>
Pero además:
<ul p align="justify">
<li>Pueden aplicarse a problemas de regresión y clasificación.
</li><br>
<li>Los árboles pueden, en teoría, manejar predictores categóricos sin tener que crear variables <code>dummy</code> o <code>one-hot-encoding</code> (no en la práctica con Scikit-learn).
</li><br>
<li>Al tratarse de métodos no paramétricos, no es necesario que se cumpla ningún tipo de distribución específica.
</li><br>
<li>Por lo general, requieren mucha menos limpieza y preprocesado de los datos en comparación a otros algoritmos de aprendizaje automático (por ejemplo, no requieren estandarización).
</li><br>
<li>Son muy útiles en la exploración de datos, permiten identificar de forma rápida y eficiente las variables más importantes. Son capaces de seleccionar variables predictoras de forma automática.
</li><br>
<li>Gracias al <code>Out-of-Bag Error</code> puede estimarse su error de validación sin necesidad de recurrir a estrategias computacionalmente costosas como la validación cruzada.
</li><br>
<li>Tienen buena escalabilidad, pueden aplicarse a conjuntos de datos con un elevado número de observaciones.</li><br>

 ## **Desventajas del Random Forest**

<p>
Sin embargo, los bosques aleatorios también tienen algunas desventajas:
<ul p align="justify">
<li><b>Sobreajuste</b>: pueden verse afectados por el sobreajuste.
</li><br>
<li><b>Lento</b>: pueden ser lentos de entrenar, especialmente cuando hay muchos árboles en el bosque. Esto se debe a que cada árbol de decisión debe entrenarse de forma independiente.
<br><br>
</ul>
<p>
En general, los bosques aleatorios son una herramienta poderosa que se puede utilizar para una variedad de tareas de ciencia de datos. Sin embargo, es importante ser consciente de sus desventajas y tomar medidas para mitigarlas.

# El modelo

## División del conjunto de datos

Se importa la función de partición y se separan predictores y clase objetivo. `stratify=y` conserva aproximadamente la proporción de clases en entrenamiento y prueba; así, la comparación de métricas no queda distorsionada por una partición accidentalmente desbalanceada.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = datos.drop(columns='clase_valor')
y = datos['clase_valor']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=123
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

## Creación del modelo

`RandomForestClassifier` implementa el bosque aleatorio para objetivos categóricos. Cada árbol aprende reglas diferentes y el conjunto agrega sus votos. `random_state` hace reproducible el experimento y `n_jobs=-1` permite utilizar los núcleos disponibles durante el ajuste.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

Se crea el clasificador con una semilla fija. Los demás hiperparámetros conservan sus valores predeterminados para presentar primero un modelo base que luego podría optimizarse mediante validación cruzada.

In [ ]:
modelo_rf = RandomForestClassifier(random_state=123, n_jobs=-1)

El método `fit()` aprende los árboles a partir de `X_train` y `y_train`. La salida textual confirma la configuración del estimador; el rendimiento todavía debe medirse sobre datos no utilizados en este ajuste.

In [ ]:
modelo_rf.fit(X_train, y_train)

## Predicción y evaluación del modelo

`predict()` asigna una clase a cada observación de prueba. Se muestran las primeras etiquetas como control y luego se evalúa el conjunto completo con exactitud, reporte de clasificación y matriz de confusión.

In [ ]:
predicciones = modelo_rf.predict(X_test)
predicciones[:10]

La **exactitud** resume la proporción total de aciertos. El reporte añade precisión, sensibilidad (`recall`) y medida F1 para cada clase. La matriz de confusión muestra los aciertos en la diagonal y permite identificar en qué dirección se producen los errores.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
exactitud = accuracy_score(y_test, predicciones)
print(f"Exactitud en prueba: {exactitud:.3f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, predicciones))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predicciones,
    cmap='Greys',
    colorbar=False
)
plt.title('Matriz de confusión en el conjunto de prueba')
plt.show()

# Importancia de los predictores

El atributo `feature_importances_` resume cuánto contribuyó cada predictor a reducir la impureza de clase en los árboles. Es útil para ordenar variables, aunque puede favorecer predictores continuos o con muchas posibilidades de corte; por eso se complementa con importancia por permutación.

In [ ]:
importancia_predictores = pd.DataFrame({
    'Predictor': X_train.columns,
    'Importancia': modelo_rf.feature_importances_.round(3)
})

In [ ]:
importancia_predictores = importancia_predictores.sort_values('Importancia', ascending=False).reset_index(drop=True)

In [ ]:
importancia_predictores

# Importancia por permutación

La importancia por permutación mide cuánto cae la exactitud cuando se desordena una variable y se rompe su relación con la clase objetivo. Una caída grande indica que el modelo dependía de ese predictor para clasificar correctamente.

La medición se realiza sobre el conjunto de prueba y se repite diez veces para estimar variabilidad. `importances_mean` resume la caída promedio de exactitud y `importances_std` su dispersión. Valores cercanos a cero indican poco aporte predictivo en este modelo.

Se importa la herramienta que recalcula el rendimiento después de permutar cada predictor. A diferencia de la importancia basada en impureza, este método evalúa directamente el efecto sobre la métrica seleccionada.

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
importancia = permutation_importance(
    estimator=modelo_rf,
    X=X_test,
    y=y_test,
    scoring='accuracy',
    n_repeats=10,
    random_state=123,
    n_jobs=-1
)

In [ ]:
df_importancia = pd.DataFrame({k: importancia[k] for k in ['importances_mean', 'importances_std']})

In [ ]:
df_importancia['Predictor'] = X_train.columns
df_importancia.sort_values('importances_mean',
                           ascending=False).reset_index(drop=True,
                                                        inplace=True)

In [ ]:
df_importancia

# Conclusiones

En este cuaderno:

- convertimos un objetivo continuo en dos clases definidas por la mediana;
- ilustramos cómo el bootstrap genera clasificadores diversos;
- combinamos etiquetas mediante voto mayoritario para construir bagging;
- entrenamos y evaluamos un `RandomForestClassifier` con una partición estratificada;
- interpretamos la matriz de confusión y dos medidas de importancia de variables.

La idea central es que la agregación de muchos árboles reduce la inestabilidad de modelos individuales y produce decisiones de clasificación más robustas.